# 1. Project Overview
**Simulated federated link prediction on the AnonNet supply-chain graph** (research prototype).

Several simulated organisations each hold the part of the supply-chain graph that belongs to
them. Each trains a GNN locally; a server averages their model *deltas* (FedAvg).

**What this notebook is, precisely**
- A *single-process simulation* of cross-silo federated learning. Clients are partitions of one
  dataset, not separate machines, and there is no network, authentication or access control.
- AnonNet has **no relation labels**, so this is *homogeneous (untyped) link prediction* — a
  KGC-style baseline, **not** relation-aware multi-relational KGC. (The LoadHalal notebook is
  the multi-relational experiment.)
- The aggregation step is a **simulation of additive pairwise masking** (arithmetic only). It is
  *not* a deployable Secure Aggregation protocol.
- **Differential privacy is disabled** in the reported runs (`DP_SIGMA = 0`), so **no formal
  privacy guarantee is claimed.** A client-level DP-FedAvg mechanism and Renyi accountant are
  provided for experimentation (section 13).

**Evaluation protocol (leakage-free)** — undirected node pairs are split *before* reverse edges
exist; node-degree features and every message-passing graph are built from *training* pairs
only; validation/test pairs (and their reverses) never enter any message-passing graph.

In [1]:
import os, io, json, math, time, copy, zlib, pickle, random, platform, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
import matplotlib.pyplot as plt

try:                      # `display` exists inside Jupyter; fall back to print elsewhere
    display
except NameError:
    display = print

try:
    import torch_geometric
    import torch_geometric.nn as pyg_nn
    import torch_geometric.utils as pyg_utils
    _HAS_PYG = True
except Exception as _e:
    _HAS_PYG = False
    print(f"torch_geometric not available ({_e}); PyG-only models will be skipped.")

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
print(f"python {platform.python_version()} | torch {torch.__version__} | "
      f"torch_geometric {'%s' % torch_geometric.__version__ if _HAS_PYG else 'n/a'}")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


python 3.12.3 | torch 2.14.1+cu130 | torch_geometric 2.8.0.post1


# 2. Configuration

In [2]:
# --- Data ---
NODE_ID_COL = None; SOURCE_COL = None; TARGET_COL = None      # None = auto-detect

def _find_dataset_dir():
    from pathlib import Path
    candidates = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd())
    candidates.extend(Path.cwd().parents)
    for base in candidates:
        if (base / "AnonNet dataset" / "AnonNet_nodes.csv").exists():
            return base
    return Path(".")

_BASE_DIR = _find_dataset_dir()
NODES_PATH = str(_BASE_DIR / "AnonNet dataset" / "AnonNet_nodes.csv")
EDGES_PATH = str(_BASE_DIR / "AnonNet dataset" / "AnonNet_edges.csv")

# --- Model / training ---
HIDDEN_DIM = 64
DROPOUT = 0.2
LEARNING_RATE = 0.01
SUP_FRAC = 0.3            # per epoch, this share of TRAIN pairs is hidden from message passing and used as
                          # supervision targets (removes the train/test mismatch of "predict edges you can see")

# --- Federation ---
NUM_CLIENTS = 4
LOCAL_EPOCHS = 3
FED_ROUNDS = 20
CENTRAL_EPOCHS = FED_ROUNDS * LOCAL_EPOCHS      # same number of gradient steps as one federated client

# --- Evaluation protocol ---
NEG_RATIO = 5             # negatives per positive for training and for AUC / PR-AUC / thresholded metrics
RANKING_CANDIDATES = 50   # filtered candidates per positive for MRR / Hits@10 (random baseline MRR ~ 0.0875, Hits@10 ~ 0.196)
SPLIT = (0.8, 0.1, 0.1)
SEEDS = [42, 43, 44]      # every number in the results table is mean +/- std over these seeds

# --- Privacy-related switches (see section 6 for what they do / do not mean) ---
PAIRWISE_MASK_SIM = True  # SIMULATED pairwise masking of client deltas (arithmetic only)
DP_SIGMA = 0.0            # DP noise multiplier. 0.0 => differential privacy is OFF (no (eps, delta) guarantee)
DP_CLIP_NORM = 1.0        # per-client delta clipping norm (used only when DP_SIGMA > 0)
DP_DELTA = 1e-5
RUN_DP_DEMO = True        # small utility-vs-epsilon illustration in section 13

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEEDS[0])
print(f"seeds={SEEDS} | central epochs={CENTRAL_EPOCHS} | fed rounds={FED_ROUNDS}x{LOCAL_EPOCHS} local epochs | "
      f"DP {'ON sigma=%g' % DP_SIGMA if DP_SIGMA > 0 else 'OFF'} | pairwise-mask simulation={'ON' if PAIRWISE_MASK_SIM else 'OFF'}")

seeds=[42, 43, 44] | central epochs=60 | fed rounds=20x3 local epochs | DP OFF | pairwise-mask simulation=ON


# 3. Load and Validate the Data

In [3]:
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_csv(path)

nodes_df = load_csv(NODES_PATH)
edges_df = load_csv(EDGES_PATH)
print("nodes:", nodes_df.shape, nodes_df.columns.tolist())
print("edges:", edges_df.shape, edges_df.columns.tolist())
display(nodes_df.head(3)); display(edges_df.head(3))

def pick(df, candidates):
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None

NODE_ID_COL = NODE_ID_COL or pick(nodes_df, ['node_id', 'nodeid', 'id', 'entity_id', 'node'])
SOURCE_COL = SOURCE_COL or pick(edges_df, ['source', 'source_id', 'from', 'head', 'subject'])
TARGET_COL = TARGET_COL or pick(edges_df, ['target', 'target_id', 'to', 'tail', 'object'])
RELATION_COL = pick(edges_df, ['relation', 'relationship', 'predicate', 'edge_type', 'type', 'label'])
assert None not in (NODE_ID_COL, SOURCE_COL, TARGET_COL), "could not detect id/source/target columns"
print(f"columns: node id={NODE_ID_COL} source={SOURCE_COL} target={TARGET_COL} relation={RELATION_COL}")
print("=> no relation column: the graph is UNTYPED (homogeneous). This is link prediction, not relation-aware KGC."
      if RELATION_COL is None else f"relation column found: {RELATION_COL}")

nodes: (7245, 3) ['id', 'location', 'products']
edges: (43926, 2) ['source', 'target']


,id,location,products
0,n0,Japan,Fertilizer
1,n1,Germany,"Feed Pump Part of Vehicle, Air Conditioner Wat..."
2,n2,Japan,Water Pump Part of Vehicle


,source,target
0,n0,n5491
1,n0,n3915
2,n0,n1494


columns: node id=id source=source target=target relation=None
=> no relation column: the graph is UNTYPED (homogeneous). This is link prediction, not relation-aware KGC.


# 4. Graph Construction — unique *undirected* pairs

The edge list is turned into a set of **unique unordered node pairs** `(u, v)` with `u < v`.
The train/validation/test split is made on these pairs, so a pair and its reverse always land in
the same split (the previous version split after adding reverse edges, which left the reverse of
~80% of "test" edges in the training graph).

In [4]:
nodes_df[NODE_ID_COL] = nodes_df[NODE_ID_COL].astype(str)
edges_df[SOURCE_COL] = edges_df[SOURCE_COL].astype(str)
edges_df[TARGET_COL] = edges_df[TARGET_COL].astype(str)

assert nodes_df[NODE_ID_COL].is_unique, "duplicate node ids"
node_list = nodes_df[NODE_ID_COL].tolist()
node2id = {n: i for i, n in enumerate(node_list)}
num_nodes = len(node_list)

total_raw = len(edges_df)
valid = edges_df[SOURCE_COL].isin(node2id) & edges_df[TARGET_COL].isin(node2id)
n_invalid = int((~valid).sum()); edges_valid = edges_df[valid]
loops = edges_valid[SOURCE_COL] == edges_valid[TARGET_COL]
n_loops = int(loops.sum()); edges_valid = edges_valid[~loops]
n_dup_dir = int(edges_valid.duplicated(subset=[SOURCE_COL, TARGET_COL]).sum())
edges_valid = edges_valid.drop_duplicates(subset=[SOURCE_COL, TARGET_COL])

s = edges_valid[SOURCE_COL].map(node2id).to_numpy(np.int64)
t = edges_valid[TARGET_COL].map(node2id).to_numpy(np.int64)
ALL_PAIRS = np.unique(np.stack([np.minimum(s, t), np.maximum(s, t)], 1), axis=0)     # [P, 2], u < v
n_reciprocal = len(edges_valid) - len(ALL_PAIRS)

print(f"nodes={num_nodes} | raw edges={total_raw} | invalid={n_invalid} | self-loops={n_loops} | "
      f"duplicate directed={n_dup_dir} | directed after cleaning={len(edges_valid)}")
print(f"reciprocal (u->v and v->u) collapsed into one undirected pair: {n_reciprocal}")
print(f"unique undirected pairs (the prediction targets): {len(ALL_PAIRS)}")
assert len(ALL_PAIRS) > 0
assert (ALL_PAIRS[:, 0] < ALL_PAIRS[:, 1]).all()

# Static, PUBLIC-per-owner node attributes only (no edge-derived information here).
def build_static_features(df, id_col, max_onehot=150):
    feats, names = [], []
    for col in df.select_dtypes(include=[np.number]).columns:
        if col == id_col: continue
        v = df[col].fillna(0).to_numpy(np.float32)
        feats.append(((v - v.mean()) / (v.std() + 1e-8)).reshape(-1, 1)); names.append(col)
    for col in df.select_dtypes(include=['object', 'category', 'string']).columns:
        if col == id_col: continue
        u = df[col].nunique()
        if u <= max_onehot:
            d = pd.get_dummies(df[col].fillna('__missing__'), prefix=col)
            feats.append(d.to_numpy(np.float32)); names += list(d.columns)
            print(f"one-hot '{col}' ({u} unique, missing -> '__missing__')")
        else:
            print(f"skipping '{col}': high cardinality ({u} unique values)")
    return torch.tensor(np.hstack(feats), dtype=torch.float32), names

STATIC_X, STATIC_NAMES = build_static_features(nodes_df, NODE_ID_COL)
IN_DIM = STATIC_X.shape[1] + 1       # + one structural feature (log-degree) computed from the *message-passing graph in use*
print(f"static feature matrix {tuple(STATIC_X.shape)}; model input dim = {IN_DIM} (log-degree is added per graph, never precomputed globally)")

nodes=7245 | raw edges=43926 | invalid=0 | self-loops=4 | duplicate directed=0 | directed after cleaning=43922
reciprocal (u->v and v->u) collapsed into one undirected pair: 2859
unique undirected pairs (the prediction targets): 41063
one-hot 'location' (105 unique, missing -> '__missing__')
skipping 'products': high cardinality (3706 unique values)
static feature matrix (7245, 106); model input dim = 107 (log-degree is added per graph, never precomputed globally)


# 5. Metrics, Sampling and Aggregation Utilities (shared with the LoadHalal notebook)

In [5]:
METRIC_NAMES = ['Threshold', 'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1-score',
                'ROC-AUC', 'PR-AUC', 'MRR', 'Hits@10']

def _np1d(a):
    if torch.is_tensor(a):
        a = a.detach().cpu().numpy()
    return np.asarray(a, dtype=np.float64).reshape(-1)

def best_f1_threshold(pos, neg):
    """Decision threshold on RAW scores that maximises F1 -- call this on
    VALIDATION scores only, then reuse the threshold unchanged on test."""
    pos, neg = _np1d(pos), _np1d(neg)
    scores = np.concatenate([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    order = np.argsort(-scores, kind='mergesort')
    s, y = scores[order], labels[order]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    prec = tp / np.maximum(tp + fp, 1)
    rec = tp / max(y.sum(), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    boundary = np.r_[np.diff(s) != 0, True]          # only cut between distinct scores
    f1 = np.where(boundary, f1, -1.0)
    k = int(np.argmax(f1))
    return float((s[k] + s[k + 1]) / 2) if k + 1 < len(s) else float(s[k] - 1e-9)

def ranking_metrics(pos, cand):
    """pos: [P] raw scores of the true pairs; cand: [P, K] raw scores of the K
    candidate corruptions of each positive. Ties get the average rank, so a
    degenerate constant scorer cannot look good."""
    pos, cand = _np1d(pos), np.asarray(cand, dtype=np.float64)
    greater = (cand > pos[:, None]).sum(1)
    ties = (cand == pos[:, None]).sum(1)
    rank = 1.0 + greater + 0.5 * ties
    return float(np.mean(1.0 / rank)), float(np.mean(rank <= 10))

def summarize_scores(pos, neg, rank_pos, rank_cand, thr):
    pos, neg = _np1d(pos), _np1d(neg)
    tp = int((pos > thr).sum()); fp = int((neg > thr).sum())
    fn, tn = len(pos) - tp, len(neg) - fp
    prec = tp / max(tp + fp, 1); rec = tp / max(len(pos), 1)
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    allsc = np.concatenate([pos, neg])
    mrr, hits10 = ranking_metrics(rank_pos, rank_cand)
    return {
        'Threshold': float(thr),
        'Accuracy': (tp + tn) / (len(pos) + len(neg)),
        'Balanced Accuracy': 0.5 * (rec + tn / max(len(neg), 1)),
        'Precision': prec, 'Recall': rec,
        'F1-score': 2 * prec * rec / max(prec + rec, 1e-12),
        'ROC-AUC': float(roc_auc_score(labels, allsc)),
        'PR-AUC': float(average_precision_score(labels, allsc)),
        'MRR': mrr, 'Hits@10': hits10,
    }

def mean_std_table(per_seed):
    """per_seed: {column_name: [metric_dict, ...]} -> DataFrame with mean / std columns."""
    rows = {'Metric': METRIC_NAMES}
    for col, runs in per_seed.items():
        arr = np.array([[r[m] for m in METRIC_NAMES] for r in runs], dtype=np.float64)
        rows[f'{col} mean'] = arr.mean(0)
        rows[f'{col} std'] = arr.std(0, ddof=1) if len(runs) > 1 else np.zeros(len(METRIC_NAMES))
    return pd.DataFrame(rows)

In [6]:
def make_key_array(pairs, num_nodes, undirected):
    """Sorted int64 keys for a set of (u, v) pairs. undirected=True makes
    (u, v) and (v, u) share one key, so reverse edges cannot slip through."""
    p = np.asarray(pairs, dtype=np.int64).reshape(-1, 2)
    if len(p) == 0:
        return np.zeros(0, dtype=np.int64)
    a, b = (np.minimum(p[:, 0], p[:, 1]), np.maximum(p[:, 0], p[:, 1])) if undirected else (p[:, 0], p[:, 1])
    return np.unique(a * num_nodes + b)

def keys_of(u, v, num_nodes, undirected):
    u, v = np.asarray(u, dtype=np.int64), np.asarray(v, dtype=np.int64)
    if undirected:
        u, v = np.minimum(u, v), np.maximum(u, v)
    return u * num_nodes + v

def in_keys(sorted_keys, q):
    if len(sorted_keys) == 0:
        return np.zeros(len(q), dtype=bool)
    idx = np.searchsorted(sorted_keys, q)
    idx[idx == len(sorted_keys)] = len(sorted_keys) - 1
    return sorted_keys[idx] == q

def sample_negative_pairs(n, rng, src_pool, dst_pool, forbidden_keys, num_nodes, undirected, max_rounds=100):
    """n distinct pairs (u in src_pool, v in dst_pool), u != v, whose key is NOT in
    forbidden_keys. Pools are explicit, so a client can only ever draw its own nodes."""
    src_pool, dst_pool = np.asarray(src_pool, dtype=np.int64), np.asarray(dst_pool, dtype=np.int64)
    got = np.zeros((0, 2), dtype=np.int64)
    for _ in range(max_rounds):
        need = n - len(got)
        if need <= 0:
            break
        m = int(need * 1.5) + 32
        u, v = rng.choice(src_pool, m), rng.choice(dst_pool, m)
        keep = u != v
        u, v = u[keep], v[keep]
        if undirected:
            u, v = np.minimum(u, v), np.maximum(u, v)
        k = u * num_nodes + v
        keep = ~in_keys(forbidden_keys, k)
        cand = np.stack([u[keep], v[keep]], 1)
        got = np.unique(np.concatenate([got, cand]), axis=0)
    if len(got) < n:
        raise RuntimeError(f"could only sample {len(got)}/{n} negatives from the given pools")
    return got[rng.permutation(len(got))[:n]]

def build_ranking_candidates(pos_pairs, k, rng, corrupt_pool, forbidden_keys, num_nodes, undirected, corrupt='tail'):
    """Filtered corruption candidates: for every positive (a, b) keep one endpoint and replace
    the other by k random nodes from `corrupt_pool`; candidates equal to ANY known positive are
    rejected (filtered protocol). corrupt='tail' always replaces b (typed KGC, e.g. which
    manufacturer supplies this food); 'either' flips a coin per positive (untyped graph).
    Returns int64 array [P, k, 2]."""
    pos = np.asarray(pos_pairs, dtype=np.int64).reshape(-1, 2)
    P = len(pos)
    keep_first = np.ones(P, dtype=bool) if corrupt == 'tail' else rng.random(P) < 0.5
    anchor = np.where(keep_first, pos[:, 0], pos[:, 1])
    A = np.repeat(anchor, k)
    B = rng.choice(np.asarray(corrupt_pool, dtype=np.int64), P * k)
    bad = np.ones(P * k, dtype=bool)
    for _ in range(200):
        idx = np.where(bad)[0]
        if len(idx) == 0:
            break
        B[idx] = rng.choice(np.asarray(corrupt_pool, dtype=np.int64), len(idx))
        bad[idx] = (A[idx] == B[idx]) | in_keys(forbidden_keys, keys_of(A[idx], B[idx], num_nodes, undirected))
    if bad.any():
        raise RuntimeError("could not build filtered ranking candidates")
    cand = np.stack([A, B], 1).reshape(P, k, 2)
    if corrupt == 'tail':
        return cand
    # untyped graph: restore (first, second) order irrelevant because scoring is symmetric
    return cand

# 6. Aggregation: simulated pairwise masking + optional client-level DP
> **Scope warning.** The code below *simulates* additive pairwise masking inside one Python
> process and tests that the masks cancel. It does **not** implement a Secure Aggregation
> protocol (no key agreement, authentication, dropout recovery, or adversary model), so it does
> **not** show that a real server could not inspect an individual client's update. DP is **off**
> unless `DP_SIGMA > 0`, and even then the accountant's assumptions (documented in
> `dp_epsilon`) are narrow. What clients send are **model deltas**, weighted by the aggregation
> weight, not raw gradients.

In [7]:
# ======================================================================
# What this cell is -- and is NOT
# ----------------------------------------------------------------------
# * PairwiseMaskSimulation is a SINGLE-PROCESS SIMULATION of additive pairwise
#   masking (the arithmetic idea behind Secure Aggregation). Each client masks
#   its own update using seeds it shares with each other client; the "server"
#   function only ever sums the masked payloads. The masks cancel exactly.
# * It is NOT a deployable Secure Aggregation protocol: there is no real key
#   agreement, no authentication, no dropout recovery and no malicious-party
#   handling, and everything runs in one Python process. It therefore does NOT
#   demonstrate that a real server cannot inspect individual client updates.
# * What is exchanged are model DELTAS (local state - global state), pre-weighted
#   by the aggregation weight.
# * Differential privacy is OFF unless DP_SIGMA > 0. When on, it is client-level
#   DP-FedAvg (clip each client delta to DP_CLIP_NORM, uniform weights, Gaussian
#   noise on the sum) with the Renyi accountant below -- see its assumptions.
# ======================================================================
def flat_norm(delta):
    return float(torch.sqrt(sum((v.double() ** 2).sum() for v in delta.values())))

def clip_delta(delta, clip_norm):
    nrm = flat_norm(delta)
    scale = min(1.0, clip_norm / (nrm + 1e-12))
    return {k: v * scale for k, v in delta.items()}, nrm

class PairwiseMaskSimulation:
    def __init__(self, num_clients, seed):
        rng = np.random.default_rng(seed + 991)
        self.num_clients = num_clients
        # secrets[i][j] is known only to clients i and j (stand-in for a key agreement).
        self._secrets = {i: {} for i in range(num_clients)}
        for i in range(num_clients):
            for j in range(i + 1, num_clients):
                s = int(rng.integers(1, 2 ** 61))
                self._secrets[i][j] = s
                self._secrets[j][i] = s

    def client_mask(self, i, payload, round_idx):
        """Runs on client i: uses ONLY client i's own pair secrets."""
        masked = {}
        for key, v in payload.items():
            m = v.double().clone()
            for j, s in self._secrets[i].items():
                seed = (s + 1_000_003 * round_idx + zlib.crc32(key.encode())) % (2 ** 62)
                g = torch.Generator().manual_seed(seed)
                r = torch.randn(v.shape, generator=g, dtype=torch.float64)
                m += r if i < j else -r
            masked[key] = m
        return masked

def server_sum(payloads):
    """Runs on the server: sees only the (masked) payloads."""
    out = {}
    for key in payloads[0]:
        out[key] = sum(p[key] for p in payloads)
    return out

def aggregate_round(client_results, round_idx, masker=None, use_masking=None, dp_sigma=None, clip_norm=None):
    """client_results: list of (delta_dict, n_samples). Returns (aggregate_delta_dict, info)."""
    use_masking = PAIRWISE_MASK_SIM if use_masking is None else use_masking
    dp_sigma = DP_SIGMA if dp_sigma is None else dp_sigma
    clip_norm = DP_CLIP_NORM if clip_norm is None else clip_norm
    K = len(client_results)
    counts = [n for _, n in client_results]
    total = sum(counts)
    if total == 0:
        return None, {}
    use_dp = dp_sigma > 0
    weights = [1.0 / K] * K if use_dp else [n / total for n in counts]
    prepared, pre_clip = [], []
    for (delta, n), w in zip(client_results, weights):
        if use_dp:
            delta, nrm = clip_delta(delta, clip_norm)
            pre_clip.append(nrm)
        prepared.append({k: v.double() * w for k, v in delta.items()})
    payloads = [masker.client_mask(i, p, round_idx) for i, p in enumerate(prepared)] if use_masking else prepared
    agg = server_sum(payloads)
    if use_dp:   # sensitivity of the uniformly-weighted sum to one client is clip_norm / K
        std = dp_sigma * clip_norm / K
        agg = {k: v + torch.randn(v.shape, dtype=torch.float64) * std for k, v in agg.items()}
    # measured size of one client's uplink message (float32 serialisation, in-process)
    uplink = len(pickle.dumps({k: v.float().numpy() for k, v in payloads[0].items()}))
    return {k: v.float() for k, v in agg.items()}, {'uplink_bytes_per_client': uplink,
                                                     'pre_clip_norms': pre_clip}

def dp_epsilon(noise_multiplier, rounds, delta):
    """(epsilon, delta) after `rounds` releases of a Gaussian mechanism with the given noise
    multiplier (std / sensitivity). Renyi DP of the Gaussian mechanism is alpha/(2 sigma^2) per
    release, composed linearly, converted with eps = RDP(alpha) + log(1/delta)/(alpha-1).
    ASSUMPTIONS: privacy unit = one client's entire local dataset (add/remove-one-client);
    every client participates every round, so there is NO subsampling amplification; noise is
    added to the aggregate by the server; model selection/hyper-parameter search is NOT
    accounted for; the validation/test graph used for scoring is outside the privacy accounting."""
    if noise_multiplier <= 0:
        return float('inf')
    alphas = np.concatenate([np.linspace(1.01, 10, 400), np.linspace(10, 4096, 800)])
    eps = rounds * alphas / (2.0 * noise_multiplier ** 2) + math.log(1.0 / delta) / (alphas - 1.0)
    return float(eps.min())

def run_privacy_simulation_tests():
    """Arithmetic checks of the SIMULATION only (not a security analysis)."""
    torch.manual_seed(0)
    shapes = {'a': (5, 3), 'b': (7,), 'c.weight': (4, 4)}
    results = [({k: torch.randn(*s) for k, s in shapes.items()}, n) for n in (10, 20, 0, 5)]
    masker = PairwiseMaskSimulation(len(results), seed=123)
    plain, _ = aggregate_round(results, 1, masker, use_masking=False, dp_sigma=0.0)
    masked, _ = aggregate_round(results, 1, masker, use_masking=True, dp_sigma=0.0)
    err = max(float((plain[k] - masked[k]).abs().max()) for k in plain)
    assert err < 1e-5, f"masked aggregate differs from plain aggregate by {err}"
    total = sum(n for _, n in results)
    expected = {k: sum(d[k] * (n / total) for d, n in results) for k in shapes}
    assert max(float((plain[k] - expected[k]).abs().max()) for k in shapes) < 1e-5
    # an individual masked payload must look nothing like the true contribution
    w0 = {k: v.double() * (results[0][1] / total) for k, v in results[0][0].items()}
    m0 = masker.client_mask(0, w0, 1)
    rel = flat_norm({k: m0[k] - w0[k] for k in w0}) / flat_norm(w0)
    assert rel > 1.0, "masked payload is too close to the true update"
    # clipping
    clipped, _ = clip_delta(results[0][0], 0.5)
    assert flat_norm(clipped) <= 0.5 + 1e-6
    # accountant sanity: more noise -> smaller epsilon; no noise -> infinite epsilon
    assert dp_epsilon(2.0, 20, 1e-5) < dp_epsilon(1.0, 20, 1e-5) < dp_epsilon(0.5, 20, 1e-5)
    assert dp_epsilon(0.0, 20, 1e-5) == float('inf')
    print(f"[simulation tests passed] max |masked - plain| = {err:.2e}; "
          f"masked/true update relative distance = {rel:.1f}")
    print("NOTE: these tests check the masking ARITHMETIC in one process; they are not a security proof.")

run_privacy_simulation_tests()

[simulation tests passed] max |masked - plain| = 0.00e+00; masked/true update relative distance = 6.3
NOTE: these tests check the masking ARITHMETIC in one process; they are not a security proof.


# 7. Models

In [8]:
class SAGEEncoder(nn.Module):
    """Two-layer mean-aggregation GraphSAGE."""
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        if _HAS_PYG:
            self.conv1 = pyg_nn.SAGEConv(in_dim, hidden_dim)
            self.conv2 = pyg_nn.SAGEConv(hidden_dim, hidden_dim)
        else:   # minimal dense fallback
            self.fc1 = nn.Linear(2 * in_dim, hidden_dim)
            self.fc2 = nn.Linear(2 * hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def _mean_agg(x, ei):
        out = torch.zeros_like(x); cnt = torch.zeros(x.size(0), 1)
        if ei.numel():
            out.index_add_(0, ei[1], x[ei[0]]); cnt.index_add_(0, ei[1], torch.ones(ei.size(1), 1))
        return out / cnt.clamp(min=1)

    def forward(self, x, ei):
        if _HAS_PYG:
            h = self.dropout(F.relu(self.conv1(x, ei)))
            return F.relu(self.conv2(h, ei))
        h = self.dropout(F.relu(self.fc1(torch.cat([x, self._mean_agg(x, ei)], 1))))
        return F.relu(self.fc2(torch.cat([h, self._mean_agg(h, ei)], 1)))

class GATEncoder(nn.Module):
    """Two-layer GAT (PyG GATConv, 4 heads in layer 1)."""
    def __init__(self, in_dim, hidden_dim, dropout, heads=4):
        super().__init__()
        self.conv1 = pyg_nn.GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = pyg_nn.GATConv(hidden_dim * heads, hidden_dim, heads=1, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, ei):
        h = self.dropout(F.elu(self.conv1(x, ei)))
        return F.elu(self.conv2(h, ei))

class LinkPredictor(nn.Module):
    """Cosine similarity x learnable temperature. Input: pairs LongTensor [E, 2]."""
    def __init__(self, init_tau=1.0):
        super().__init__()
        self.tau = nn.Parameter(torch.tensor(float(init_tau)))

    def forward(self, z, pairs):
        assert pairs.dim() == 2 and pairs.size(1) == 2, f"pairs must be [E, 2], got {tuple(pairs.shape)}"
        z = F.normalize(z, p=2, dim=-1)
        return (z[pairs[:, 0]] * z[pairs[:, 1]]).sum(-1) * self.tau

class KGCModel(nn.Module):
    def __init__(self, kind, in_dim, hidden_dim, dropout):
        super().__init__()
        self.kind = kind
        self.encoder = SAGEEncoder(in_dim, hidden_dim, dropout) if kind == 'GraphSAGE' else GATEncoder(in_dim, hidden_dim, dropout)
        self.predictor = LinkPredictor()

KINDS = ['GraphSAGE'] + (['GAT'] if _HAS_PYG else [])
def make_model(kind):
    return KGCModel(kind, IN_DIM, HIDDEN_DIM, DROPOUT)
print("models:", KINDS, "| parameters:", {k: sum(p.numel() for p in make_model(k).parameters()) for k in KINDS})

models: ['GraphSAGE', 'GAT'] | parameters: {'GraphSAGE': 22017, 'GAT': 44737}


# 8. Graph helpers, training step, evaluation
`make_x` computes the structural feature (log-degree) **from the message-passing graph that is
actually passed to the encoder** — the training graph (with supervision pairs hidden), a client's
local graph, or the full training graph at evaluation. Held-out pairs can therefore never
influence a feature.

In [9]:
def bidir(pairs):
    """LongTensor [E, 2] of undirected pairs -> edge_index [2, 2E] with both directions."""
    return torch.cat([pairs.T, pairs.flip(1).T], dim=1)

def make_x(static_x, edge_index, num_nodes):
    deg = torch.bincount(edge_index[0], minlength=num_nodes).float() if edge_index.numel() else torch.zeros(num_nodes)
    return torch.cat([static_x, torch.log1p(deg).unsqueeze(1)], dim=1)

def train_step(model, opt, static_x, num_nodes, pairs, neg_pool, gen):
    """One full-batch step. A random SUP_FRAC of `pairs` is hidden from message passing and used as
    positive supervision; the rest forms the message-passing graph (+ reverse directions)."""
    E = pairs.size(0)
    if E < 2:
        return float('nan')
    model.train(); opt.zero_grad()
    perm = torch.randperm(E, generator=gen)
    n_sup = max(1, int(round(SUP_FRAC * E)))
    sup, msg = pairs[perm[:n_sup]], pairs[perm[n_sup:]]
    ei = bidir(msg)
    z = model.encoder(make_x(static_x, ei, num_nodes), ei)
    neg = neg_pool[torch.randint(0, neg_pool.size(0), (NEG_RATIO * n_sup,), generator=gen)]
    ps, ns = model.predictor(z, sup), model.predictor(z, neg)
    loss = F.binary_cross_entropy_with_logits(ps, torch.ones_like(ps)) + \
           F.binary_cross_entropy_with_logits(ns, torch.zeros_like(ns))
    loss.backward(); opt.step()
    return float(loss.detach())

@torch.no_grad()
def evaluate_model(model, ctx, split, thr=None):
    """Scores `split` ('val' | 'test') on the FULL TRAINING graph. The threshold for the thresholded
    metrics is tuned on validation scores and reused unchanged for test."""
    model.eval()
    ei = bidir(ctx['train_pairs'])
    z = F.normalize(model.encoder(make_x(ctx['static_x'], ei, ctx['num_nodes']), ei), dim=-1)
    tau = model.predictor.tau
    sc = lambda p: ((z[p[:, 0]] * z[p[:, 1]]).sum(-1) * tau).cpu().numpy()
    b = ctx[split]
    pos, neg = sc(b['pos']), sc(b['neg'])
    P, K = b['cand'].shape[:2]
    cand = sc(b['cand'].reshape(-1, 2)).reshape(P, K)
    thr = best_f1_threshold(pos, neg) if thr is None else thr
    return summarize_scores(pos, neg, pos, cand, thr)

def degree_product_baseline(ctx):
    """NON-LEARNED reference (preferential attachment): score(u, v) = log(1+deg_u) + log(1+deg_v) on the TRAIN graph.
    Uniform random negatives favour low-degree nodes, so degree alone can score well; the GNN must beat this."""
    deg = np.bincount(ctx['train_pairs'].numpy().reshape(-1), minlength=ctx['num_nodes'])
    sc = lambda p: (np.log1p(deg[p[:, 0].numpy()]) + np.log1p(deg[p[:, 1].numpy()])).astype(np.float64)
    def run(split, thr=None):
        b = ctx[split]; pos, neg = sc(b['pos']), sc(b['neg']); P, K = b['cand'].shape[:2]
        cand = sc(b['cand'].reshape(-1, 2)).reshape(P, K)
        return summarize_scores(pos, neg, pos, cand, best_f1_threshold(pos, neg) if thr is None else thr)
    return run('test', run('val')['Threshold'])

# 9. Federated Partition (by geographic region) and Per-Experiment Preparation
Ownership is defined by the node attribute `location` (not by edges), so it cannot encode held-out
edge information. A client sees **only**: its own nodes' static attributes, edges whose *both*
endpoints it owns, negatives drawn from *its own* node set, and degree features computed from its
own local graph. `assert_client_isolation` checks this mechanically.

In [10]:
node_locations = nodes_df['location'].fillna('Unknown').astype(str).to_numpy()
unique_locations = sorted(set(node_locations))
location_to_client = {loc: i % NUM_CLIENTS for i, loc in enumerate(unique_locations)}
NODE_TO_CLIENT = np.array([location_to_client[l] for l in node_locations])
for c in range(NUM_CLIENTS):
    print(f"Client {c+1}: {(NODE_TO_CLIENT == c).sum()} nodes in {sum(1 for l in unique_locations if location_to_client[l] == c)} regions")

def assert_no_heldout_in_graph(edge_index, heldout_pairs, name):
    """No held-out pair, in EITHER direction, may exist in a message-passing graph."""
    ei = edge_index.numpy().T
    g = make_key_array(ei, num_nodes, undirected=True)
    h = make_key_array(heldout_pairs, num_nodes, undirected=True)
    assert not np.isin(h, g).any(), f"LEAK: held-out pair present in {name}"

def assert_client_isolation(client, expected_local_edges):
    n = client['num_nodes']
    assert client['static_x'].shape[0] == n, "client holds features for nodes it does not own"
    for name in ('pairs', 'neg_pool'):
        t = client[name]
        assert t.ndim == 2 and t.shape[1] == 2, f"{name} must be [E, 2]"
        assert t.numel() == 0 or (int(t.min()) >= 0 and int(t.max()) < n), f"{name} references a node outside the client"
        owners = NODE_TO_CLIENT[client['nodes_global'][t.numpy().reshape(-1)]]
        assert (owners == client['client_id']).all(), f"{name} contains an endpoint owned by another client"
    assert client['pairs'].size(0) == expected_local_edges

def build_clients(ctx, seed):
    tr = ctx['train_pairs'].numpy()
    clients = []
    for c in range(NUM_CLIENTS):
        nodes_c = np.where(NODE_TO_CLIENT == c)[0]
        g2l = -np.ones(num_nodes, dtype=np.int64); g2l[nodes_c] = np.arange(len(nodes_c))
        mask = (NODE_TO_CLIENT[tr[:, 0]] == c) & (NODE_TO_CLIENT[tr[:, 1]] == c)
        local_pairs = g2l[tr[mask]]                                        # LOCAL ids
        rng = np.random.default_rng(seed * 100 + 10 + c)
        forb = make_key_array(local_pairs, len(nodes_c), True)
        pool_n = max(NEG_RATIO * len(local_pairs) * 2, 16)
        neg = sample_negative_pairs(pool_n, rng, np.arange(len(nodes_c)), np.arange(len(nodes_c)), forb, len(nodes_c), True)
        client = {'client_id': c, 'nodes_global': nodes_c, 'num_nodes': len(nodes_c),
                  'static_x': STATIC_X[nodes_c].clone(),                    # ONLY this client's rows
                  'pairs': torch.LongTensor(local_pairs), 'neg_pool': torch.LongTensor(neg)}
        assert_client_isolation(client, int(mask.sum()))
        clients.append(client)
    return clients

def prepare(seed):
    """Everything for one seed: split -> negatives -> ranking candidates -> clients, with leakage assertions."""
    rng = np.random.default_rng(seed)
    P = len(ALL_PAIRS); perm = rng.permutation(P)
    n_tr, n_va = int(SPLIT[0] * P), int(SPLIT[1] * P)
    train, val, test = ALL_PAIRS[perm[:n_tr]], ALL_PAIRS[perm[n_tr:n_tr + n_va]], ALL_PAIRS[perm[n_tr + n_va:]]

    k_all = make_key_array(ALL_PAIRS, num_nodes, True)
    k_tr, k_va, k_te = (make_key_array(a, num_nodes, True) for a in (train, val, test))
    assert not (np.isin(k_tr, k_va).any() or np.isin(k_tr, k_te).any() or np.isin(k_va, k_te).any()), "splits overlap"
    assert len(k_tr) + len(k_va) + len(k_te) == len(k_all), "split does not partition the pair set"

    allnodes = np.arange(num_nodes)
    # training negatives avoid TRAIN positives and use only nodes seen in training; evaluation negatives avoid ALL known positives
    train_nodes = np.unique(train.reshape(-1))      # nodes seen in TRAIN pairs only
    train_neg = sample_negative_pairs(NEG_RATIO * len(train) * 2, rng, train_nodes, train_nodes, k_tr, num_nodes, True)
    def bundle(pairs):
        neg = sample_negative_pairs(NEG_RATIO * len(pairs), rng, allnodes, allnodes, k_all, num_nodes, True)
        cand = build_ranking_candidates(pairs, RANKING_CANDIDATES, rng, allnodes, k_all, num_nodes, True, 'either')
        assert not in_keys(k_all, make_key_array(neg, num_nodes, True)).any()
        return {'pos': torch.LongTensor(pairs), 'neg': torch.LongTensor(neg), 'cand': torch.LongTensor(cand)}
    ctx = {'seed': seed, 'num_nodes': num_nodes, 'static_x': STATIC_X, 'train_pairs': torch.LongTensor(train),
           'train_neg_pool': torch.LongTensor(train_neg), 'val': bundle(val), 'test': bundle(test)}

    # ---- acceptance tests: no held-out pair (or its reverse) in ANY message-passing graph ----
    heldout = np.concatenate([val, test])
    assert_no_heldout_in_graph(bidir(ctx['train_pairs']), heldout, "the evaluation message-passing graph")
    sup_perm = torch.randperm(len(train), generator=torch.Generator().manual_seed(seed))
    assert_no_heldout_in_graph(bidir(ctx['train_pairs'][sup_perm[int(SUP_FRAC * len(train)):]]), heldout, "a training message-passing graph")
    # sensitivity check: the test WOULD fire if a held-out pair's reverse were present
    leaky = torch.cat([ctx['train_pairs'], torch.LongTensor(test[:1]).flip(1)])
    fired = False
    try:
        assert_no_heldout_in_graph(bidir(leaky), heldout, "deliberately leaky graph")
    except AssertionError:
        fired = True
    assert fired, "leak test is insensitive"

    ctx['clients'] = build_clients(ctx, seed)
    c2c = lambda a: float((NODE_TO_CLIENT[a[:, 0]] != NODE_TO_CLIENT[a[:, 1]]).mean())
    ctx['stats'] = {'train_pairs': len(train), 'val_pairs': len(val), 'test_pairs': len(test),
                    'cross_client_train': c2c(train), 'cross_client_val': c2c(val), 'cross_client_test': c2c(test),
                    'local_train_edges_per_client': [int(c['pairs'].size(0)) for c in ctx['clients']]}
    return ctx

ctx0 = prepare(SEEDS[0])
st = ctx0['stats']
print(f"split (seed {SEEDS[0]}): train={st['train_pairs']} val={st['val_pairs']} test={st['test_pairs']} undirected pairs")
print(f"cross-client share  train={st['cross_client_train']:.1%}  val={st['cross_client_val']:.1%}  test={st['cross_client_test']:.1%}")
print(f"=> clients train only on the {1 - st['cross_client_train']:.1%} intra-client pairs; cross-client pairs are EXCLUDED from federated training "
      f"(the federated model never learns from them), yet they are part of the evaluation.")
print("local training pairs per client:", st['local_train_edges_per_client'])
print("leakage acceptance tests passed: disjoint splits, no held-out pair or reverse in any message graph, client isolation OK")

Client 1: 1899 nodes in 27 regions
Client 2: 437 nodes in 27 regions
Client 3: 2179 nodes in 26 regions
Client 4: 2730 nodes in 26 regions


split (seed 42): train=32850 val=4106 test=4107 undirected pairs
cross-client share  train=51.2%  val=51.5%  test=51.8%
=> clients train only on the 48.8% intra-client pairs; cross-client pairs are EXCLUDED from federated training (the federated model never learns from them), yet they are part of the evaluation.
local training pairs per client: [1496, 492, 4802, 9250]
leakage acceptance tests passed: disjoint splits, no held-out pair or reverse in any message graph, client isolation OK


# 10. Centralized and Federated Training
Both settings use the same number of gradient steps per client (`CENTRAL_EPOCHS = FED_ROUNDS x LOCAL_EPOCHS`),
select the best checkpoint on **validation ROC-AUC**, and report test metrics with a threshold tuned on
validation only. The federated server scores validation/test pairs on the global training graph — this is an
*evaluation harness*; in a real deployment no single party would hold that graph.

In [11]:
def fit_central(kind, ctx, seed):
    set_seed(seed)
    model = make_model(kind); opt = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    gen = torch.Generator().manual_seed(seed)
    best, hist = {'auc': -1.0}, []
    for ep in range(1, CENTRAL_EPOCHS + 1):
        loss = train_step(model, opt, ctx['static_x'], ctx['num_nodes'], ctx['train_pairs'], ctx['train_neg_pool'], gen)
        if ep % LOCAL_EPOCHS == 0:
            m = evaluate_model(model, ctx, 'val')
            hist.append({'step': ep // LOCAL_EPOCHS, 'loss': loss, 'val_auc': m['ROC-AUC'], 'val_mrr': m['MRR']})
            if m['ROC-AUC'] > best['auc']:
                best = {'auc': m['ROC-AUC'], 'state': copy.deepcopy(model.state_dict()), 'step': ep // LOCAL_EPOCHS}
    model.load_state_dict(best['state'])
    thr = evaluate_model(model, ctx, 'val')['Threshold']
    return {'test': evaluate_model(model, ctx, 'test', thr), 'val_auc': best['auc'], 'best_step': best['step'], 'history': hist}

def client_update(kind, global_state, client, seed, rnd):
    """Runs on the client: sees only its own static features / local graph / local negatives."""
    model = make_model(kind); model.load_state_dict(global_state)
    opt = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    gen = torch.Generator().manual_seed(seed * 100003 + rnd * 101 + client['client_id'])
    n = client['pairs'].size(0)
    if n >= 2:
        for _ in range(LOCAL_EPOCHS):
            train_step(model, opt, client['static_x'], client['num_nodes'], client['pairs'], client['neg_pool'], gen)
    else:
        n = 0
    new = model.state_dict()
    delta = {k: (new[k] - global_state[k]).float() for k in global_state}     # model DELTA, not raw gradients
    return delta, n

def fit_federated(kind, ctx, seed, dp_sigma=None, clip_norm=None):
    set_seed(seed)
    global_model = make_model(kind)
    masker = PairwiseMaskSimulation(NUM_CLIENTS, seed)
    best, hist, uplink = {'auc': -1.0}, [], []
    for rnd in range(1, FED_ROUNDS + 1):
        gstate = {k: v.clone() for k, v in global_model.state_dict().items()}
        results = [client_update(kind, gstate, c, seed, rnd) for c in ctx['clients']]
        agg, info = aggregate_round(results, rnd, masker, dp_sigma=dp_sigma, clip_norm=clip_norm)
        if agg is not None:
            global_model.load_state_dict({k: gstate[k] + agg[k] for k in gstate})
            uplink.append(info['uplink_bytes_per_client'])
        m = evaluate_model(global_model, ctx, 'val')
        hist.append({'step': rnd, 'val_auc': m['ROC-AUC'], 'val_mrr': m['MRR'], 'val_f1': m['F1-score'], 'val_prauc': m['PR-AUC'], 'val_hits10': m['Hits@10']})
        if m['ROC-AUC'] > best['auc']:
            best = {'auc': m['ROC-AUC'], 'state': copy.deepcopy(global_model.state_dict()), 'step': rnd}
    global_model.load_state_dict(best['state'])
    thr = evaluate_model(global_model, ctx, 'val')['Threshold']
    return {'test': evaluate_model(global_model, ctx, 'test', thr), 'val_auc': best['auc'], 'best_step': best['step'],
            'history': hist, 'uplink_bytes_per_client_round': float(np.mean(uplink)) if uplink else 0.0,
            'num_params': sum(p.numel() for p in global_model.parameters())}
print("training functions defined")

training functions defined


In [12]:
BASE = 'Degree-product baseline (non-learned)'
RESULTS = {f'{k} ({s})': [] for k in KINDS for s in ('Centralized', 'Federated')}
RESULTS[BASE] = []
HISTORY, COMM, STATS_PER_SEED, RUN_LOG = {}, {}, {}, []
t_all = time.time()
for seed in SEEDS:
    ctx = ctx0 if seed == SEEDS[0] else prepare(seed)
    STATS_PER_SEED[seed] = ctx['stats']
    RESULTS[BASE].append(degree_product_baseline(ctx))
    print(f"seed {seed} | degree-product baseline (non-learned): AUC={RESULTS[BASE][-1]['ROC-AUC']:.4f} MRR={RESULTS[BASE][-1]['MRR']:.4f}")
    for kind in KINDS:
        t0 = time.time()
        cen = fit_central(kind, ctx, seed)
        fed = fit_federated(kind, ctx, seed)
        RESULTS[f'{kind} (Centralized)'].append(cen['test']); RESULTS[f'{kind} (Federated)'].append(fed['test'])
        if seed == SEEDS[0]:
            HISTORY[kind] = {'central': cen['history'], 'federated': fed['history']}
        COMM.setdefault(kind, []).append((fed['num_params'], fed['uplink_bytes_per_client_round']))
        RUN_LOG.append({'seed': seed, 'model': kind, 'central_best_step': cen['best_step'], 'fed_best_round': fed['best_step']})
        print(f"seed {seed} | {kind:9s} | central AUC={cen['test']['ROC-AUC']:.4f} MRR={cen['test']['MRR']:.4f} | "
              f"federated AUC={fed['test']['ROC-AUC']:.4f} MRR={fed['test']['MRR']:.4f} | {time.time() - t0:.0f}s")
print(f"total time {time.time() - t_all:.0f}s")

seed 42 | degree-product baseline (non-learned): AUC=0.9320 MRR=0.3957


seed 42 | GraphSAGE | central AUC=0.8725 MRR=0.3361 | federated AUC=0.8390 MRR=0.3055 | 13s


seed 42 | GAT       | central AUC=0.8785 MRR=0.4029 | federated AUC=0.8452 MRR=0.3575 | 37s


seed 43 | degree-product baseline (non-learned): AUC=0.9323 MRR=0.4109


seed 43 | GraphSAGE | central AUC=0.8826 MRR=0.3366 | federated AUC=0.8510 MRR=0.3301 | 14s


seed 43 | GAT       | central AUC=0.8826 MRR=0.3995 | federated AUC=0.8560 MRR=0.3681 | 37s


seed 44 | degree-product baseline (non-learned): AUC=0.9313 MRR=0.4065


seed 44 | GraphSAGE | central AUC=0.8811 MRR=0.3803 | federated AUC=0.8259 MRR=0.3335 | 11s


seed 44 | GAT       | central AUC=0.8798 MRR=0.4003 | federated AUC=0.8509 MRR=0.3711 | 36s
total time 152s


# 11. Results (mean ± std over seeds)

In [13]:
summary = mean_std_table(RESULTS)
per_seed_rows = []
for col, runs in RESULTS.items():
    for seed, r in zip(SEEDS, runs):
        per_seed_rows.append({'model_setting': col, 'seed': seed, **r})
pd.DataFrame(per_seed_rows).to_csv('centralized_vs_federated_per_seed.csv', index=False)
summary.to_csv('centralized_vs_federated.csv', index=False)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
display(summary)
pool_note = (f"Protocol: {len(SEEDS)} seeds; validation-tuned F1 threshold; class balance 1 positive : {NEG_RATIO} negatives "
             f"(majority-class accuracy = {NEG_RATIO / (1 + NEG_RATIO):.3f}); ranking = filtered either-endpoint corruption with "
             f"{RANKING_CANDIDATES} candidates per positive (random baseline MRR ~ {np.mean(1 / np.arange(1, RANKING_CANDIDATES + 2)):.4f}, "
             f"Hits@10 = {10 / (RANKING_CANDIDATES + 1):.4f}).")
print(pool_note)

hist_df = pd.DataFrame(HISTORY['GraphSAGE']['federated']).rename(columns={'step': 'round'})
hist_df.rename(columns={'val_auc': 'auc', 'val_f1': 'f1', 'val_prauc': 'prauc', 'val_mrr': 'mrr', 'val_hits10': 'hits10'}).to_csv('federated_history.csv', index=False)
print("saved centralized_vs_federated.csv, centralized_vs_federated_per_seed.csv, federated_history.csv")

,Metric,GraphSAGE (Centralized) mean,GraphSAGE (Centralized) std,GraphSAGE (Federated) mean,GraphSAGE (Federated) std,GAT (Centralized) mean,GAT (Centralized) std,GAT (Federated) mean,GAT (Federated) std,Degree-product baseline (non-learned) mean,Degree-product baseline (non-learned) std
0,Threshold,0.7070,0.1246,0.9983,0.2246,0.9473,0.0286,1.0619,0.1652,5.6801,0.0165
1,Accuracy,0.8828,0.0056,0.8575,0.0143,0.8589,0.0026,0.8637,0.0050,0.8995,0.0012
2,Balanced Accuracy,0.8336,0.0053,0.7704,0.0173,0.8039,0.0088,0.7837,0.0035,0.8165,0.0021
3,Precision,0.6217,0.0161,0.5653,0.0426,0.5595,0.0076,0.5799,0.0151,0.7012,0.0047
4,Recall,0.7597,0.0059,0.6398,0.0279,0.7215,0.0224,0.6637,0.0036,0.6920,0.0043
5,F1-score,0.6838,0.0115,0.5998,0.0312,0.6301,0.0077,0.6189,0.0089,0.6966,0.0032
6,ROC-AUC,0.8787,0.0054,0.8386,0.0126,0.8803,0.0021,0.8507,0.0054,0.9319,0.0005
7,PR-AUC,0.5729,0.0153,0.5840,0.0370,0.5873,0.0052,0.5992,0.0198,0.7741,0.0023
8,MRR,0.3510,0.0254,0.3230,0.0153,0.4009,0.0018,0.3656,0.0071,0.4044,0.0078
9,Hits@10,0.7444,0.0251,0.6795,0.0187,0.7854,0.0027,0.7373,0.0090,0.7320,0.0092


Protocol: 3 seeds; validation-tuned F1 threshold; class balance 1 positive : 5 negatives (majority-class accuracy = 0.833); ranking = filtered either-endpoint corruption with 50 candidates per positive (random baseline MRR ~ 0.0886, Hits@10 = 0.1961).
saved centralized_vs_federated.csv, centralized_vs_federated_per_seed.csv, federated_history.csv


## Reference-paper numbers (indicative only — **not** a like-for-like comparison)
The reference paper (Peeris et al., *Sustainability* 2026, 18, 2825, Table 5) reports centralized
GraphSAGE/GAT on AnonNet. Its candidate pool, negative-sampling rule and split are not matched
here, so thresholded metrics, MRR and Hits@10 are **not directly comparable**. ROC-AUC is the least
protocol-sensitive number, but differences in splitting still apply.

In [14]:
paper = pd.DataFrame({
    'Method': ['Paper GraphSAGE', 'Paper GAT'],
    'Accuracy': [0.6740, 0.8610], 'ROC-AUC': [0.8587, 0.9112], 'F1': [0.4349, 0.7254],
    'MRR': [0.0436, 0.0477], 'Hits@10': [0.4000, 0.4200]})
display(paper)
ours = summary.set_index('Metric')
rows = []
for k in KINDS:
    rows.append({'Method': f'This work {k} (centralized)', 'Accuracy': ours.loc['Accuracy', f'{k} (Centralized) mean'],
                 'ROC-AUC': ours.loc['ROC-AUC', f'{k} (Centralized) mean'], 'F1': ours.loc['F1-score', f'{k} (Centralized) mean'],
                 'MRR': ours.loc['MRR', f'{k} (Centralized) mean'], 'Hits@10': ours.loc['Hits@10', f'{k} (Centralized) mean']})
display(pd.DataFrame(rows))

,Method,Accuracy,ROC-AUC,F1,MRR,Hits@10
0,Paper GraphSAGE,0.6740,0.8587,0.4349,0.0436,0.4000
1,Paper GAT,0.8610,0.9112,0.7254,0.0477,0.4200


,Method,Accuracy,ROC-AUC,F1,MRR,Hits@10
0,This work GraphSAGE (centralized),0.8828,0.8787,0.6838,0.3510,0.7444
1,This work GAT (centralized),0.8589,0.8803,0.6301,0.4009,0.7854


# 12. Communication: measured serialisation vs theoretical estimate
`uplink` is the **measured size of one client's serialised float32 delta** in this process. It is
**not** network traffic: it excludes transport/encryption overhead, mask/key exchange, retries and
dropouts, none of which exist in this simulation.

In [15]:
comm_rows = []
for kind in KINDS:
    params = COMM[kind][0][0]
    measured = float(np.mean([m for _, m in COMM[kind]]))
    comm_rows.append({'model': kind, 'parameters': params, 'theoretical_payload_MB(4B/param)': params * 4 / 2 ** 20,
                      'measured_serialised_uplink_MB/client/round': measured / 2 ** 20,
                      'estimated_total_uplink_MB (clients x rounds)': measured * NUM_CLIENTS * FED_ROUNDS / 2 ** 20})
comm_df = pd.DataFrame(comm_rows); display(comm_df)
print("Downlink (broadcast of the global model) is not counted; these are payload sizes, not measured network traffic.")

,model,parameters,theoretical_payload_MB(4B/param),measured_serialised_uplink_MB/client/round,estimated_total_uplink_MB (clients x rounds)
0,GraphSAGE,22017,0.0840,0.0845,6.7610
1,GAT,44737,0.1707,0.1713,13.7032


Downlink (broadcast of the global model) is not counted; these are payload sizes, not measured network traffic.


# 13. Differential-privacy illustration (optional, GraphSAGE, first seed)
This shows the *mechanism and accountant*, not a privacy claim for the main results (which use
`DP_SIGMA = 0`). Privacy unit = one client's whole local dataset; with only 4 clients per round
there is no subsampling amplification and the noise needed for a small epsilon swamps the signal.

In [16]:
DP_DEMO = []
if RUN_DP_DEMO:
    base = RESULTS['GraphSAGE (Federated)'][0]
    DP_DEMO.append({'dp_sigma': 0.0, 'epsilon': float('inf'), 'ROC-AUC': base['ROC-AUC'], 'MRR': base['MRR']})
    for sigma in (0.1, 0.5, 1.0, 2.0):
        r = fit_federated('GraphSAGE', ctx0, SEEDS[0], dp_sigma=sigma, clip_norm=DP_CLIP_NORM)
        DP_DEMO.append({'dp_sigma': sigma, 'epsilon': dp_epsilon(sigma, FED_ROUNDS, DP_DELTA),
                        'ROC-AUC': r['test']['ROC-AUC'], 'MRR': r['test']['MRR']})
    dp_df = pd.DataFrame(DP_DEMO); display(dp_df)
    dp_df.to_csv('dp_demo_results.csv', index=False)
    print(f"epsilon is at delta={DP_DELTA}, {FED_ROUNDS} rounds, client-level, all clients every round, no subsampling. "
          "Rows with a large epsilon give no meaningful privacy guarantee. The main results above use DP OFF.")

,dp_sigma,epsilon,ROC-AUC,MRR
0,0.0000,inf,0.8390,0.3055
1,0.1000,1215.1105,0.8333,0.3042
2,0.5000,82.9245,0.7594,0.2737
3,1.0000,31.4598,0.7393,0.2725
4,2.0000,13.2299,0.7401,0.2728


epsilon is at delta=1e-05, 20 rounds, client-level, all clients every round, no subsampling. Rows with a large epsilon give no meaningful privacy guarantee. The main results above use DP OFF.


# 14. Plots

In [17]:
os.makedirs('plots', exist_ok=True)
COL = {'GraphSAGE': '#2c7fb8', 'GAT': '#d95f0e'}

def _despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 1 -- federated validation curves (first seed)
fig, ax1 = plt.subplots(figsize=(9, 4.6), dpi=120); ax2 = ax1.twinx()
for kind in KINDS:
    h = pd.DataFrame(HISTORY[kind]['federated'])
    ax1.plot(h['step'], h['val_auc'], color=COL[kind], lw=2.2, marker='o', ms=3.5, label=f'{kind} val ROC-AUC')
    ax2.plot(h['step'], h['val_mrr'], color=COL[kind], lw=1.6, ls='--', marker='s', ms=3, label=f'{kind} val MRR')
ax1.set_xlabel('Federated round'); ax1.set_ylabel('Validation ROC-AUC'); ax2.set_ylabel('Validation MRR (dashed)')
ax1.grid(alpha=.25, ls='--')
l1, n1 = ax1.get_legend_handles_labels(); l2, n2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, n1 + n2, loc='lower right', fontsize=8)
ax1.set_title(f'Federated training (FedAvg, simulated masking, seed {SEEDS[0]})', fontweight='bold')
fig.tight_layout(); fig.savefig('plots/fed_training_history.png', dpi=150, bbox_inches='tight'); plt.close(fig)

# 2 -- model comparison with seed-level error bars
cols = [f'{k} ({s})' for k in KINDS for s in ('Centralized', 'Federated')]
bar_col = {'GraphSAGE (Centralized)': '#2c7fb8', 'GraphSAGE (Federated)': '#7fcdbb', 'GAT (Centralized)': '#d95f0e', 'GAT (Federated)': '#fdb863'}
sm = summary.set_index('Metric')
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), dpi=120)
for ax, metric in zip(axes, ['ROC-AUC', 'PR-AUC', 'MRR']):
    means = [sm.loc[metric, f'{c} mean'] for c in cols]; stds = [sm.loc[metric, f'{c} std'] for c in cols]
    bars = ax.bar(range(len(cols)), means, yerr=stds, capsize=4, color=[bar_col[c] for c in cols], edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c.replace(' (', '\n(') for c in cols], fontsize=8)
    ax.set_title(f'{metric} (mean ± std, {len(SEEDS)} seeds)', fontweight='bold'); ax.grid(axis='y', alpha=.3, ls='--'); _despine(ax)
    if metric == 'ROC-AUC': ax.axhline(0.5, color='grey', ls='--', lw=.8)
fig.tight_layout(); fig.savefig('plots/model_comparison.png', dpi=150, bbox_inches='tight'); plt.close(fig)

# 3 -- client data distribution
edges_pc = ctx0['stats']['local_train_edges_per_client']
nodes_pc = [int((NODE_TO_CLIENT == c).sum()) for c in range(NUM_CLIENTS)]
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.4), dpi=120)
cc = ['#2c7fb8', '#41b6c4', '#7fcdbb', '#d9ef8b']
a1.bar(range(1, NUM_CLIENTS + 1), nodes_pc, color=cc, edgecolor='white'); a1.set_title('Nodes owned per client', fontweight='bold')
a2.bar(range(1, NUM_CLIENTS + 1), edges_pc, color=cc, edgecolor='white'); a2.set_title('Local (intra-client) training pairs', fontweight='bold')
sh = [ctx0['stats'][k] for k in ('cross_client_train', 'cross_client_val', 'cross_client_test')]
a3.bar(['train', 'val', 'test'], sh, color='#d95f0e', edgecolor='white'); a3.set_ylim(0, 1); a3.set_title('Cross-client share of pairs\n(never used in federated training)', fontweight='bold')
for ax in (a1, a2, a3): ax.grid(axis='y', alpha=.3, ls='--'); _despine(ax)
for ax in (a1, a2): ax.set_xlabel('Client')
fig.tight_layout(); fig.savefig('plots/client_data_distribution.png', dpi=150, bbox_inches='tight'); plt.close(fig)

# 4 -- communication (measured serialised payload; no network)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4), dpi=120)
x = np.arange(len(comm_df)); w = .35
a1.bar(x - w/2, comm_df['theoretical_payload_MB(4B/param)'], w, label='theoretical (4 B/param)', color='#2c7fb8')
a1.bar(x + w/2, comm_df['measured_serialised_uplink_MB/client/round'], w, label='measured pickle size', color='#d95f0e')
a1.set_xticks(x); a1.set_xticklabels(comm_df['model']); a1.set_ylabel('MB per client per round'); a1.legend(fontsize=8)
a1.set_title('Client uplink payload', fontweight='bold')
for kind in KINDS:
    per_round = comm_df.set_index('model').loc[kind, 'measured_serialised_uplink_MB/client/round'] * NUM_CLIENTS
    a2.plot(range(1, FED_ROUNDS + 1), np.cumsum(np.full(FED_ROUNDS, per_round)), color=COL[kind], lw=2.2, label=kind)
a2.set_xlabel('Federated round'); a2.set_ylabel('Cumulative uplink (MB, all clients)'); a2.legend(); a2.set_title('Cumulative uplink payload (estimate)', fontweight='bold')
for ax in (a1, a2): ax.grid(alpha=.3, ls='--'); _despine(ax)
fig.tight_layout(); fig.savefig('plots/communication_analysis.png', dpi=150, bbox_inches='tight'); plt.close(fig)

# 5 -- dataset structure (undirected-pair degree distribution + locations)
deg = np.bincount(ALL_PAIRS.reshape(-1), minlength=num_nodes)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4), dpi=120)
vals, cnts = np.unique(deg, return_counts=True)
a1.loglog(vals[vals > 0], cnts[vals > 0], 'o', ms=3.5, color='#2c7fb8'); a1.set_xlabel('Degree (undirected pairs)'); a1.set_ylabel('Nodes')
a1.set_title('Degree distribution (log-log)', fontweight='bold'); a1.grid(alpha=.3, ls='--'); _despine(a1)
loc = pd.Series(node_locations).value_counts().head(12)
a2.barh(range(len(loc)), loc.values, color='#d95f0e', edgecolor='white'); a2.set_yticks(range(len(loc))); a2.set_yticklabels(loc.index, fontsize=8)
a2.invert_yaxis(); a2.set_title('Top 12 locations (incl. "Unknown" = missing)', fontweight='bold'); a2.grid(axis='x', alpha=.3, ls='--'); _despine(a2)
fig.tight_layout(); fig.savefig('plots/dataset_structure.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print("plots written to plots/")

plots written to plots/


# 15. Node classification: synthetic labels only (**no classifier is trained here**)
The reference paper also describes node classification on AnonNet using *synthetic* carbon scores
(low/medium/high risk). This cell only generates such synthetic labels. They are a deterministic
function of `location` plus noise, so they are **not** real sustainability data, and **no node-classification result is
reported by this project.**

In [18]:
def generate_carbon_scores(df, n, seed):
    rng = np.random.default_rng(seed)
    intensity = {'China': 1.00, 'India': 0.85, 'Japan': 0.55, 'South Korea': 0.60, 'United States': 0.70,
                 'Germany': 0.45, 'United Kingdom': 0.40, 'Taiwan': 0.65, 'Indonesia': 0.90, 'Australia': 0.60, 'Thailand': 0.75}
    locs = df['location'].fillna('Unknown').astype(str).tolist()
    return np.array([intensity.get(l, 0.65) * rng.lognormal(sigma=0.35) for l in locs], dtype=np.float32)

carbon = generate_carbon_scores(nodes_df, num_nodes, SEEDS[0])
p33, p67 = np.percentile(carbon, [33.33, 66.67])
risk = np.where(carbon <= p33, 0, np.where(carbon <= p67, 1, 2))
print("synthetic risk-band counts (Low/Medium/High):", np.bincount(risk, minlength=3).tolist(), "-- illustrative only")

synthetic risk-band counts (Low/Medium/High): [2415, 2415, 2415] -- illustrative only


# 16. Run metadata (reproducibility record)

In [19]:
meta = {
    'notebook': 'anonnet_federated_kg_completion.ipynb',
    'run_timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(), 'torch': torch.__version__,
    'torch_geometric': torch_geometric.__version__ if _HAS_PYG else None,
    'numpy': np.__version__, 'pandas': pd.__version__,
    'seeds': SEEDS,
    'config': {k: globals()[k] for k in ['HIDDEN_DIM', 'DROPOUT', 'LEARNING_RATE', 'SUP_FRAC', 'NUM_CLIENTS', 'LOCAL_EPOCHS',
                                        'FED_ROUNDS', 'CENTRAL_EPOCHS', 'NEG_RATIO', 'RANKING_CANDIDATES', 'SPLIT',
                                        'PAIRWISE_MASK_SIM', 'DP_SIGMA', 'DP_CLIP_NORM', 'DP_DELTA']},
    'privacy_status': {'differential_privacy': 'OFF (DP_SIGMA=0): no (epsilon, delta) guarantee in the reported runs',
                       'secure_aggregation': 'simulated pairwise masking in one process; not a deployable protocol'},
    'evaluation_protocol': pool_note,
    'data': {'nodes': num_nodes, 'undirected_pairs': int(len(ALL_PAIRS)), 'relation_column': RELATION_COL},
    'per_seed_split_stats': {str(k): v for k, v in STATS_PER_SEED.items()},
    'leakage_acceptance_tests': 'passed in prepare() for every seed (disjoint splits; no held-out pair/reverse in any message graph; client isolation)',
}
with open('run_metadata_anonnet.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(json.dumps({k: meta[k] for k in ['run_timestamp_utc', 'torch', 'torch_geometric', 'seeds']}, indent=1))

{
 "run_timestamp_utc": "2026-10-13T03:44:48+00:00",
 "torch": "2.14.1+cu130",
 "torch_geometric": "2.8.0.post1",
 "seeds": [
  42,
  43,
  44
 ]
}


# 17. Interpretation and Claims

## What the results can support
- Comparison of *centralized* vs *simulated-federated* training of GraphSAGE/GAT for **homogeneous link
  prediction** on AnonNet, under a leakage-free protocol, as mean ± std over several seeds.
- A non-learned degree-product baseline is reported next to the GNNs; read the GNN numbers relative to it, because uniform random negatives
  favour low-degree nodes and degree alone separates a lot.
- The federated models learn only from intra-client pairs; the measured gap to centralized training
  therefore reflects *both* the loss of cross-client structure *and* the feature/degree distribution shift between local
  and global graphs. This notebook does not separate the two effects.

## What the results do **not** support
- **No privacy guarantee.** DP is off. The masking is a one-process arithmetic simulation, so it is not evidence that
  a real server cannot inspect individual updates; it is not a Secure Aggregation implementation.
- **No deployment claim.** Clients are simulated partitions of one dataset in one process.
- **No claim that the GNNs learn more than degree structure.** In the reported run the non-learned degree-product baseline matched or exceeded the
  GNNs on ROC-AUC and PR-AUC (see the results table). With uniform random negatives, a degree prior is very strong on this graph; the GNNs
  are not shown to add predictive value beyond it.
- **No relation-aware KGC claim.** AnonNet has no relation labels.
- **No claim of beating the reference paper**: protocols differ (candidate pools, negatives, splits).
- Ranking metrics depend on the sampled candidate pool (here 50 filtered candidates) and are not comparable
  across papers using other pools. Thresholded metrics use a threshold tuned on validation data;
  with 1 : 5 class balance the majority-class accuracy is already 0.833, so look at balanced accuracy, ROC-AUC and PR-AUC.

## Known limitations
- Cross-client pairs (see section 9 for the share) are excluded from federated training.
- Region-based ownership gives clients with very different sizes and a large `Unknown`-location block.
- Degree features computed on a client's local graph have a different scale from degrees on the global training graph.
- Communication numbers are payload sizes, not measured network traffic.

## Future work (not implemented)
Real Secure Aggregation inside a federated framework; calibrated DP with larger client counts; relation labels for
multi-relational GNNs; learned cross-silo link models; measured multi-process communication.